## Part 2 Statistical Text Representations
### Muhammad Athallah Yakarazi (24/532752/PA/22532)



#### Import python libraries

In [ ]:
# part 2 text preprocessing

# Muhammad Athallah Yakarazi (24/532752/PA/22532)

import pandas as pd
import re # built-in regex library
import math

#### Load corpus into pandas dataframe and apply corpus preprocessing

In [ ]:
file_name = 'spam.csv'
df = pd.read_csv(file_name, encoding='latin-1')
#print(df.head())

# Common english stopwords https://gist.github.com/sebleier/554280
# I chose the stopwords listed there because it saves me work of having to manually gather them myself and they are already comprehensive
STOP_WORDS = {
 'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 'yourself',
 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its', 'itself',
 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that',
 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had',
 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as',
 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through',
 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off',
 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should',
 'now'
}

def stemmer(word):
    if word.endswith('ies') and len(word) > 4:
        return word[:-3] + 'y'  # e.g., 'cities' -> 'city'
    elif word.endswith('es') and len(word) > 3:
        return word[:-2]        # e.g., 'foxes' -> 'fox'
    elif word.endswith('s') and len(word) > 2:
        return word[:-1]        # e.g., 'cats' -> 'cat'
    elif word.endswith('ing') and len(word) > 4:
        return word[:-3]        # e.g., 'running' -> 'runn'
    elif word.endswith('ed') and len(word) > 3:
        return word[:-2]        # e.g., 'walked' -> 'walk'
    elif word.endswith('ly') and len(word) > 3:
        return word[:-2]        # e.g., 'badly' -> 'bad'
    return word

def preprocess_corpus(text):
    if not isinstance(text, str):
        return []
    # Lowercase, remove punctuation marks and numbers
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenisation then stopword removal then stemming
    tokens = text.split()
    processed_words = [word for word in tokens if word not in STOP_WORDS]
    stemmed_processed_words = [stemmer(word) for word in processed_words]

    return stemmed_processed_words

df['processed_tokens'] = df['v2'].fillna('').apply(preprocess_corpus)
#print(df['processed_tokens'].iloc[[69]])
#print(df['processed_tokens'].head())

nth_row = 166 # My handpicked document in the dataset for specific document scorings

#### Bag of Words representation

In [ ]:
# Bag of Words
# Vocabulary list
vocabulary = {}
for tokens in df['processed_tokens']:
    for token in tokens:
        if token not in vocabulary:
            vocabulary[token] = len(vocabulary)
#print(vocabulary)

# Create bag of words vector adapted from https://www.datacamp.com/tutorial/python-bag-of-words-model
def create_bow_vector(tokens , vocab):
    vector = [0] * len(vocab)  # Init a vector of zeros
    for word in tokens:
        if word in vocab:
            idx = vocab.index(word)  # Find the index of the word in the vocabulary
            vector[idx] += 1  # Increment the count at that index

    return vector

# Apply the function to create a new column with the vectors
def apply_create_bow_vector(tokens):
    return create_bow_vector(tokens, list(vocabulary.keys()))

df['bow_vector'] = df['processed_tokens'].apply(apply_create_bow_vector)
#print(df['bow_vector'].head())

# Dimensionality
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Vector Length: {len(df['bow_vector'])}")
print(f"Dimensionality: {len(df['bow_vector'][0])}")

# Reverse the vocabulary mapping (index -> word) so it can be numerically ranked for the highest weighted terms under TF-IDF later
index_to_word = {index: word for word, index in vocabulary.items()}

# 10 Highest weighted terms under BoW
# Retrieve BoW vector for the specific document then pair words with their counts if count more than 0
doc_bow_vector = df['bow_vector'].iloc[nth_row]
bow_term_scores = [
    (index_to_word[idx], count) for idx, count in enumerate(doc_bow_vector) if count > 0
]

# Sort descending by count and select the top 10
top_10_bow = sorted(bow_term_scores, key=lambda x: x[1], reverse=True)[:10]

print(f"\nTop 10 BoW scores for document {nth_row}:")
print(f"{'Term'}, {'BoW Count'}")
for word, count in top_10_bow:
    print(f"{word:}, {count}")

Vocabulary size: 7409
Vector Length: 5572
Dimensionality: 7409

Top 10 BoW scores for document 166:
Term, BoW Count
prize, 1
claim, 1
call, 1
code, 1
valid, 1
urgent, 1
won, 1
try, 1
weekend, 1
guarante, 1


#### TF-IDF representation

In [ ]:
# TF-IDF

N = len(df) # Total number of document

# Init document frequencies
document_frequencies = {}
for word in vocabulary:
    document_frequencies[word] = 0 # start with 0 documents are containing that word

# Count of how many documents contain each word
for tokens in df['processed_tokens']:
    unique_words_in_doc = set(tokens) # because set do not allow duplicate values
    # print(unique_words_in_doc)
    for word in unique_words_in_doc:
        if word in document_frequencies:
            document_frequencies[word] += 1
#print(document_frequencies)

# IDF for each word
idf_dictionary = {}
for word, doc_count in document_frequencies.items():
    idf_dictionary[word] = math.log((N + 1) / (doc_count + 1)) # Smoothed IDF

def create_tfidf_vector(tokens, vocab, idf_dict):
    # Init a vector of zeros matching the vocabulary size
    vector = [0.0] * len(vocab)
    total_tokens = len(tokens)

    # Return early if the document is empty to prevent divide by zero error
    if total_tokens == 0:
        return vector

    # Calculate TF for the current document
    term_counts = {}
    for word in tokens:
        term_counts[word] = term_counts.get(word, 0) + 1

    # Calculate the final TF-IDF score for each word in the document
    for word, count in term_counts.items():
        if word in vocab:
            # Normalised TF: word count divided by total words in the document then place the score at the correct index in the vector
            tf = count / total_tokens
            tfidf_score = tf * idf_dict[word]
            vector[vocab[word]] = tfidf_score

    return vector

# Apply the function to create a new column with the vectors
def apply_create_tfidf_vector(tokens):
    return create_tfidf_vector(tokens, vocabulary, idf_dictionary)

df['tfidf_vector'] = df['processed_tokens'].apply(apply_create_tfidf_vector)
#print(df['tfidf_vector'].head())

# Dimensionality
print(f"Vector Length: {len(df['tfidf_vector'])}")
print(f"Dimensionality: {len(df['tfidf_vector'][0])}")

# 10 Highest under TF-IDF
doc_tfidf_vector = df['tfidf_vector'].iloc[nth_row] # Retrieve

# Pair words with their TF-IDF scores filter out 0.0 scores
tfidf_term_scores = [
    (index_to_word[idx], score) for idx, score in enumerate(doc_tfidf_vector) if score > 0.0
]

# Sort descending by the TF-IDF score and select the top 10
top_10_tfidf = sorted(tfidf_term_scores, key=lambda x: x[1], reverse=True)[:10]

print(f"\nTop 10 TF-IDF scores for document {nth_row}:")
print(f"{'Term'}, {'TF-IDF Score'}")
for word, score in top_10_tfidf:
    print(f"{word:}, {score:.4f}")

Vector Length: 5572
Dimensionality: 7409

Top 10 TF-IDF scores for document 166:
Term, TF-IDF Score
valid, 0.3632
code, 0.3461
weekend, 0.3291
hr, 0.3291
draw, 0.3228
guarante, 0.3129
urgent, 0.2968
show, 0.2918
won, 0.2881
contact, 0.2863


#### N-Gram representation

In [ ]:
# N-Gram

def generate_ngrams(tokens, n):
  ngrams = []
  for i in range(len(tokens) - n + 1): # Stop loop n earlier to prevent out of bounds error
    # Group consecutive overlapping words joined by a space
    ngram = ' '.join(tokens[i:i+n])
    ngrams.append(ngram)

  return ngrams

# Here I generated Trigram because it is more suitable for spam detection (i.e., "you have won")
def apply_generate_ngrams(tokens):
  return generate_ngrams(tokens, 3)

df['ngrams'] = df['processed_tokens'].apply(apply_generate_ngrams)
#print(df['processed_tokens'].tail())
#print(df['ngrams'].head())
#print(df['ngrams'].tail())

# Build vocab for ngram
ngram_vocabulary = {}
for ngram_list in df['ngrams']:
    for ngram in ngram_list:
        if ngram not in ngram_vocabulary:
            ngram_vocabulary[ngram] = len(ngram_vocabulary)
#print(ngram_vocabulary)
#print(len(ngram_vocabulary))

def create_ngram_vector(ngrams, vocab):
    # Init a list of zeros matching the vocabulary size
    vector = [0] * len(vocab)

    # Count frequencies for the current document
    for ngram in ngrams:
        if ngram in vocab:
            vector[vocab[ngram]] += 1

    return vector

# Apply the function to create a new column with the vectors
def apply_create_ngram_vector(ngrams):
    return create_ngram_vector(ngrams, ngram_vocabulary)

df['ngram_vector'] = df['ngrams'].apply(apply_create_ngram_vector)
#print(df['ngram_vector'].head())

# Dimensionality
print(f"Corpus Vocabulary size: {len(vocabulary)}")
print(f"N-Gram Vocabulary size: {len(ngram_vocabulary)}")

Corpus Vocabulary size: 7409
N-Gram Vocabulary size: 31859


#### Cosine similarity on similar documents

In [ ]:
def calculate_cosine_similarity(vector_a, vector_b):
    if len(vector_a) != len(vector_b):
        raise ValueError('Vector length not equal')

    #print(vector_a)
    #print(vector_b)

    dot_product = 0.0
    magnitudesum_a = 0.0
    magnitudesum_b = 0.0

    # Dot product = a1*b1 + a2*b2 + a3*b3 + ...
    # Magnitudesum = a1^2 + a2^2 + a3^2 + ...
    for a, b in zip(vector_a, vector_b):
        dot_product += a * b
        magnitudesum_a += a ** 2
        magnitudesum_b += b ** 2

    # Prevent division by zero if a document has no words in the vocabulary
    if magnitudesum_a == 0.0 or magnitudesum_b == 0.0:
        return 0.0

    return dot_product / (math.sqrt(magnitudesum_a) * math.sqrt(magnitudesum_b))

# Convert lists of tokens back into strings so it can be fed into the calculate function
def tokens_to_string(tokens):
    return ' '.join(tokens) if isinstance(tokens, list) else ''

df['processed_tokens_str'] = df['processed_tokens'].apply(tokens_to_string)


# Similar documents 166 and 187
print(df['processed_tokens_str'].iloc[166])
print(df['processed_tokens_str'].iloc[187])

doc1_tfidf_vector = df['tfidf_vector'].iloc[166] # Retrieve
doc2_tfidf_vector = df['tfidf_vector'].iloc[187]
doc1_bow_vector = df['bow_vector'].iloc[166]
doc2_bow_vector = df['bow_vector'].iloc[187]

cosine_sim_bow = calculate_cosine_similarity(doc1_bow_vector, doc2_bow_vector)
cosine_sim_tfidf = calculate_cosine_similarity(doc1_tfidf_vector, doc2_tfidf_vector)

print('Similar documents')
print('Cosine Similarity (BoW):', cosine_sim_bow)
print('Cosine Similarity (TF-IDF):', cosine_sim_tfidf)

urgent try contact last weekend draw show won prize guarante call claim code valid hr
please call customer service representative freephone ampm won guarante cash prize
Similar documents
Cosine Similarity (BoW): 0.3113995776646092
Cosine Similarity (TF-IDF): 0.22906811627542525


#### Cosine similarity on disimilar documents

In [ ]:
# Disimilar documents 333 and 248
print(df['processed_tokens_str'].iloc[333])
print(df['processed_tokens_str'].iloc[248])

doc1_tfidf_vector = df['tfidf_vector'].iloc[333]
doc2_tfidf_vector = df['tfidf_vector'].iloc[248]
doc1_bow_vector = df['bow_vector'].iloc[333]
doc2_bow_vector = df['bow_vector'].iloc[248]

cosine_sim_bow = calculate_cosine_similarity(doc1_bow_vector, doc2_bow_vector)
cosine_sim_tfidf = calculate_cosine_similarity(doc1_tfidf_vector, doc2_tfidf_vector)

print('Disimilar documents')
print('Cosine Similarity (BoW):', cosine_sim_bow)
print('Cosine Similarity (TF-IDF):', cosine_sim_tfidf)

chance might evaporat soon violat privacy steal phone number employer paperwork cool please contact report supervisor
didnt work oh ok goodnight ill fix ready time wake dear miss good night sleep
Disimilar documents
Cosine Similarity (BoW): 0.0
Cosine Similarity (TF-IDF): 0.0
